<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "768"
iso_country = "Togo"
year = 2022
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_DX.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 8.650*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Togo': 

Number of trade records present in dataset: 6720
Number of partner countries present in dataset: 115
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1908.9m
Total weight within the dataset: 2308.4 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,768,Togo,156,China,Import,Sea,551349,"551349 | Fabrics, woven; prin",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,123463.136,100534.312,123463.136,kg,130019.0,130019.0,19025.669258,20245.0,12579.655895,239336.372433,['Container'],['Container'],742.058432,739.168323,739.381429,739.381429,741.231736,741.231736,780.364879,1480.861495,864.978575,790.963029,794.202790,819.755171,762.257983,1406.261906,899.545891,825.159193,806.342753,847.724539,746.383808,1556.892040,1126.649841,982.465427,907.620139,886.471822,730.733993,1646.393801,1319.532246,1232.407690,968.229101,954.539846,715.377533,1695.573889,1582.558439,1606.363961,1050.510482,1004.731879,699.728872,1655.994441,1614.071270,1703.273021,1272.065076,1023.553513,684.672226,1476.230297,1514.163161,1619.886381,1223.775990,1036.107438,669.713924,1310.031125,1348.592006,1586.480344,1283.180341,1044.145649,2676
1,1,768,Togo,410,Rep. of Korea,Import,Sea,551512,"551512 | Fabrics, woven; of p",5515,5515 | Woven fabrics of syn,55,55 | Man-made staple fibr,2679,Other woven fabrics of man-made staple fibres,Textiles and Wearing Apparel,27,TEX - Textiles,24852.275,19881.821,24852.275,kg,25206.0,25206.0,2937.538905,20916.0,12996.595836,38178.005901,['Container'],['Container'],118.370271,117.909252,117.943246,117.943246,118.238399,118.238399,124.480766,236.221258,137.978013,126.171342,126.688136,130.764152,121.592424,224.321422,143.492057,131.626180,128.624655,135.225716,119.060238,248.349354,179.718794,156.719058,144.780029,141.406532,116.563840,262.626326,210.486644,196.588874,154.448135,152.264478,114.114238,270.471342,252.443558,256.240922,167.573340,160.270916,111.618024,264.157783,257.470362,271.699478,202.914866,163.273270,109.216247,235.482507,241.533410,258.397966,195.211979,165.275823,106.830157,208.971062,215.122144,253.069165,204.687930,166.558047,2679


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $297.8m (15.6%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
10 | Cereals               79240125.4  190956779.1      4.2      8.3
15 | Animal, vegetable or  51190604.9   59092079.0      2.7      2.6
17 | Sugars and sugar con  33989167.9   64537458.7      1.8      2.8
22 | Beverages, spirits a  22265997.1   16583449.6      1.2      0.7
19 | Preparations of cere  21702798.4   12262216.9      1.1      0.5
03 | Fish and crustaceans  19443202.4    9292340.6      1.0      0.4
04 | Dairy produce; birds  15200124.8    7766428.7      0.8      0.3
21 | Miscellaneous edible  15193114.1    4212363.0      0.8      0.2
02 | Meat and edible meat  10940446.8    5978577.8      0.6      0.3
11 | Products of the mill   9126280.4   17043626.5      0.5      0.7
20 | Preparations of vege   8065916.9    7275508.5      0.4      0.3
16

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $257.3m (13.5%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       142516932.8  185761705.9      7.5      8.0
Food & Beverages  112681819.5  104328838.1      5.9      4.5
Fishing             2063972.9     956753.4      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $222.4m (11.7%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     63518234.3  38346547.6      3.3      1.7
VOL - Vegetable oils and fats  51039531.1  59106985.6      2.7      2.6
PCR - Processed rice           26730176.1  46430315.9      1.4      2.0
SGR - Sugar                    24871420.6  54562922.4      1.3      2.4
PDR - Paddy rice               24412118.1  42403849.2      1.3      1.8
MIL - Dairy products           15269190.5   7685489.3      0.8      0.3
OMT - Meat products n.e.c.     12290151.9   6205869.7      0.6      0.3
V_F - Vegetables, fruit, nuts   2434329.8   4877304.4      0.1      0.2
OCR - Crops n.e.c.              1093824.3    415909.1      0.1      0.0
GRO - Cereal grains n.e.c.       371974.8   1493498.6      0.0      0.1
CMT - Bovine meat prods          313511.9    107211.5      0.0      0.

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 15.6% 
GTAP 11.7% 
EORA 13.5%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46431/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,768,Togo,156,China,Import,Sea,551349,"551349 | Fabrics, woven; prin",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,123463.136,100534.312,123463.136,kg,130019.0,130019.0,19025.669258,20245.0,12579.655895,239336.372433,['Container'],['Container'],742.058432,739.168323,739.381429,739.381429,741.231736,741.231736,780.364879,1480.861495,864.978575,790.963029,794.202790,819.755171,762.257983,1406.261906,899.545891,825.159193,806.342753,847.724539,746.383808,1556.892040,1126.649841,982.465427,907.620139,886.471822,730.733993,1646.393801,1319.532246,1232.407690,968.229101,954.539846,715.377533,1695.573889,1582.558439,1606.363961,1050.510482,1004.731879,699.728872,1655.994441,1614.071270,1703.273021,1272.065076,1023.553513,684.672226,1476.230297,1514.163161,1619.886381,1223.775990,1036.107438,669.713924,1310.031125,1348.592006,1586.480344,1283.180341,1044.145649,2676,NaN
1,1,768,Togo,410,Rep. of Korea,Import,Sea,551512,"551512 | Fabrics, woven; of p",5515,5515 | Woven fabrics of syn,55,55 | Man-made staple fibr,2679,Other woven fabrics of man-made staple fibres,Textiles and Wearing Apparel,27,TEX - Textiles,24852.275,19881.821,24852.275,kg,25206.0,25206.0,2937.538905,20916.0,12996.595836,38178.005901,['Container'],['Container'],118.370271,117.909252,117.943246,117.943246,118.238399,118.238399,124.480766,236.221258,137.978013,126.171342,126.688136,130.764152,121.592424,224.321422,143.492057,131.626180,128.624655,135.225716,119.060238,248.349354,179.718794,156.719058,144.780029,141.406532,116.563840,262.626326,210.486644,196.588874,154.448135,152.264478,114.114238,270.471342,252.443558,256.240922,167.573340,160.270916,111.618024,264.157783,257.470362,271.699478,202.914866,163.273270,109.216247,235.482507,241.533410,258.397966,195.211979,165.275823,106.830157,208.971062,215.122144,253.069165,204.687930,166.558047,2679,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $190.7m of $1908.9m (10.0%)
Volume: 311.2 of 2308.4 million kilograms (13.5%)
Value-to-Weight: 0.61 from 0.83 US$ per kg (+-25.9%)
Average distance to market (volume-weighted): 6272.2 from 6312.7 nm (+-0.6%)

            food_group  N_com    cifvalue   vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    177  96199958.8  122703029.2      5.0      5.3
1      Sugar and Honey     36  33989167.9   64537458.7      1.8      2.8
2  Unprocessed Cereals     14  28097831.2  102122614.0      1.5      4.4
3                Dairy     57  15096718.6    7731529.8      0.8      0.3
4        Meat and Prep     46  10825633.2    5968818.8      0.6      0.3
5        Fats and Oils     34   3282626.7    2666062.2      0.2      0.1
6           Vegetables     46   2429314.1    4687419.2      0.1      0.2
7               Fruits     18    733499.1     747674.1      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,14106448.6,26740509.7,15631206.4,14296762.2,14355278.1,14817244.4,12933921.1,30627799.2,28584868.7,29019312.2,18982370.6,18155385.4,12109991.0,23666317.4,24362676.5,28654416.7,23183004.4,18866622.3
1,Fats and Oils,5581082.7,10337047.3,6147103.4,5642907.0,5664850.8,5850735.5,5137478.3,11806949.6,11059976.2,11255141.9,7446934.9,7114229.2,4825988.2,9215489.1,9487280.6,11137912.5,9042253.5,7383150.0
2,Fruits,1087421.4,2056582.1,1204189.8,1101889.4,1106380.6,1141994.3,997424.5,2355738.7,2198919.3,2232963.9,1462059.2,1398247.8,934188.9,1821979.1,1875491.0,2205149.4,1784545.7,1452888.0
3,Meat and Prep,8563509.8,16250567.2,9492037.2,8679811.0,8715363.2,8995768.0,7850356.5,18606762.0,17366561.5,17627796.8,11528013.4,11025651.6,7349256.6,14375921.6,14799078.1,17409599.4,14081268.5,11458167.5
4,Processed Cereals,329135543.2,595136158.9,359740955.5,332001611.0,333274390.3,344251319.5,304225991.8,688846316.3,646535086.2,659421210.0,440893440.0,414884907.0,286742610.7,540268075.3,554312906.7,646329574.3,528872439.9,429687229.3
5,Sugar and Honey,87829306.8,162278420.6,96595968.2,88784081.5,89134790.9,92039981.3,80887287.5,186898178.6,174965582.0,178074049.4,117883005.2,111762431.7,76012270.8,145598577.2,149614491.7,175151949.0,142570260.0,115929703.8
6,Unprocessed Cereals,171604897.5,325359433.2,190162227.8,173919858.5,174631381.0,180252371.8,157338316.8,372605762.9,347805216.4,353067984.8,230988967.5,220857508.0,147314036.4,287960391.7,296418590.3,348650052.4,282055223.3,229507044.5
7,Vegetables,8017840.2,15163225.8,8878566.6,8124348.2,8157477.2,8420144.7,7354394.4,17370202.2,16214964.1,16466034.3,10781124.0,10309345.9,6888081.2,13433332.8,13827842.0,16258748.2,13158565.8,10712019.7


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.84,0.10,0.01,0.02,0.05,1.17,1.04,1.07,0.40,0.35,0.77,0.81,1.10,0.73,0.45
1,Fats and Oils,1.45,0.17,0.02,0.03,0.08,2.03,1.80,1.86,0.70,0.60,1.34,1.42,1.92,1.28,0.78
2,Fruits,1.32,0.16,0.02,0.03,0.07,1.85,1.64,1.68,0.63,0.55,1.21,1.28,1.73,1.16,0.71
3,Meat and Prep,0.71,0.09,0.01,0.01,0.04,0.99,0.88,0.90,0.34,0.29,0.65,0.69,0.93,0.62,0.38
4,Processed Cereals,2.77,0.32,0.03,0.04,0.16,4.00,3.56,3.69,1.42,1.15,2.64,2.78,3.74,2.52,1.49
5,Sugar and Honey,2.19,0.26,0.03,0.04,0.12,3.12,2.77,2.86,1.09,0.91,2.05,2.17,2.92,1.96,1.17
6,Unprocessed Cereals,5.47,0.66,0.08,0.11,0.31,7.66,6.78,6.97,2.62,2.26,5.01,5.31,7.17,4.80,2.93
7,Vegetables,2.94,0.35,0.04,0.06,0.17,4.12,3.65,3.75,1.41,1.22,2.69,2.86,3.86,2.58,1.57
